In [ ]:
# =============================================================================
# Classical Simulation of Heisenberg Model on a SINGLE HEXAGON (N=6)
# Initial State: Qubit 0 in |1> (Spin Down), others in |0>
# Plotting: Probability of Qubit 0 being in |1> (Spin Down Occupation)
# =============================================================================

import numpy as np
import scipy.linalg
from scipy.linalg import expm # Matrix exponentiation for exp(-iHt)
# --- Import for Smoothing ---
from scipy.interpolate import UnivariateSpline
# --------------------------
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm # Use notebook version for Jupyter
import networkx as nx # Needed for lattice generation

# --- Qiskit Imports (needed ONLY to generate matrix representations) ---
# These imports are used here as convenient tools to define quantum operators
# and initial states, which are then converted to matrices/vectors for
# classical simulation using SciPy. They are not used for quantum simulation itself.
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.circuit import Parameter # Import Parameter if definition uses it


def compute_H_sparse_heisenberg_graph( # More general function
    graph: nx.Graph,
    J: float | Parameter,
) -> tuple[SparsePauliOp, dict, int]: # Return H, node_map, num_qubits
    """
    Generates Heisenberg XXX (XX+YY+ZZ) Hamiltonian for a given graph.

    Args:
        graph: A networkx graph defining connectivity.
        J: Uniform coupling constant.

    Returns:
        A tuple containing:
        - H (SparsePauliOp): The Hamiltonian operator.
        - node_to_index (dict): Mapping from networkx nodes to qubit indices (0 to N-1).
        - num_qubits (int): The total number of qubits (sites).
    """
    num_qubits = graph.number_of_nodes()
    if num_qubits == 0:
        return SparsePauliOp("", coeffs=[0.0]), {}, 0

    # Determine mapping from networkx nodes to qubit indices (0 to N-1).
    # NetworkX can use various node types; we need a consistent integer index.
    first_node = list(graph.nodes())[0]
    if isinstance(first_node, int) and set(graph.nodes()) == set(range(num_qubits)):
         # If nodes are already 0-based integers, use them directly.
         print("Using graph nodes directly as qubit indices (0 to N-1).")
         node_to_index = {i: i for i in range(num_qubits)}
    else:
         # Otherwise, create a map by sorting nodes.
         print("Nodes are not 0 to N-1 integers. Creating index map...")
         sorted_nodes = sorted(list(graph.nodes())) # Sort nodes for consistent mapping
         node_to_index = {node: i for i, node in enumerate(sorted_nodes)}

    pauli_list = []; coeffs_list = []
    identity_list = list("I" * num_qubits)
    term_coeff = J

    print(f"Building Hamiltonian for {num_qubits}-site graph...")

    for u, v in graph.edges():
        idx_u = node_to_index[u] # Get integer qubit index for node u
        idx_v = node_to_index[v] # Get integer qubit index for node v

        # Qiskit's SparsePauliOp and Statevector typically use little-endian indexing
        # for string labels and bitstrings, where index 0 corresponds to qubit N-1,
        # index 1 to qubit N-2, ..., index N-1 to qubit 0.
        # So, to place an operator on qubit k (0 to N-1), the character goes at index N-1-k.
        op_string_idx_u = num_qubits - 1 - idx_u
        op_string_idx_v = num_qubits - 1 - idx_v


        if not (0 <= op_string_idx_u < num_qubits and 0 <= op_string_idx_v < num_qubits):
             print(f"Warning: Calculated Pauli string indices {op_string_idx_u}, {op_string_idx_v} out of bounds.")
             continue

        # XX term
        xx = identity_list[:]; xx[op_string_idx_u], xx[op_string_idx_v] = 'X', 'X'
        pauli_list.append("".join(xx)); coeffs_list.append(term_coeff)
        # YY term
        yy = identity_list[:]; yy[op_string_idx_u], yy[op_string_idx_v] = 'Y', 'Y'
        pauli_list.append("".join(yy)); coeffs_list.append(term_coeff)
        # ZZ term
        zz = identity_list[:]; zz[op_string_idx_u], zz[op_string_idx_v] = 'Z', 'Z'
        pauli_list.append("".join(zz)); coeffs_list.append(term_coeff)

    if not pauli_list: H = SparsePauliOp("I" * num_qubits, coeffs=[0.0]) # Zero Hamiltonian if no edges
    else: H = SparsePauliOp(pauli_list, coeffs=coeffs_list)
    return H.simplify(atol=1e-12), node_to_index, num_qubits


def generate_single_number_operator_op(num_qubits: int, site_index: int) -> SparsePauliOp:
    """
    Generates the number operator for a specific site (qubit index).
    This operator is defined as n_k = (|1><1|)_k, which is equivalent to (I - Z_k)/2.
    Its expectation value <psi|n_k|psi> gives the probability of finding site k in the |1> (spin down) state.
    """
    if not 0 <= site_index < num_qubits: raise ValueError(f"site_index {site_index} out of bounds [0, {num_qubits-1}]")
    if num_qubits == 0: return SparsePauliOp("", coeffs=[0.0])

    identity_op = SparsePauliOp("I" * num_qubits)
    pauli_list_z = list("I" * num_qubits)
    # Index in the Pauli string corresponding to the target qubit (site_index)
    op_string_index = num_qubits - 1 - site_index # Qiskit string index for qubit 'site_index'
    if not 0 <= op_string_index < num_qubits: raise IndexError("Invalid Pauli string index calculation.")
    pauli_list_z[op_string_index] = 'Z'
    Z_k = SparsePauliOp("".join(pauli_list_z)) # Pauli-Z operator on qubit site_index

    # Number operator n_k = (|1><1|)_k = (I - Z_k)/2
    n_k = 0.5 * (identity_op - Z_k)
    return n_k.simplify(atol=1e-12) # Simplify to combine/clean up terms


# --- Simulation Parameters ---
# System Definition: Single Hexagon (N=6)
num_sites_hexagon = 6
# Explicitly defines the expected number of sites for a single hexagon graph.

# Physics Parameters
J_val = 1.0            # Heisenberg coupling constant J 

# --- Site Selection (Indices 0 to N-1 for the system) ---
# Note: These are qubit indices, mapping to graph nodes via compute_H_sparse_heisenberg_graph.
initial_excited_site_idx = 0 # Qubit index where the initial state is |1> (spin down)
plot_site_idx = 0          # Qubit index whose probability of being |1> (spin down) is plotted

# --- Time Parameters ---
t_max = 6.25               # Maximum simulation time (units of 1/J)
num_time_points = 100      # Number of time points to sample between 0 and t_max
times = np.linspace(0, t_max, num_time_points) # Array of time points


# --- Generate Operators and Matrices ---
# This section builds the Hamiltonian and the operator for the observable (number operator)
# and converts them into dense matrices for classical simulation.
print("Generating Heisenberg Hamiltonian for a SINGLE HEXAGON (N=6)...")
# Status message.
# Using networkx.cycle_graph(6) produces a 6-node graph (0 to 5) forming a hexagon.
hexagon_graph = nx.cycle_graph(num_sites_hexagon)
# Generates a cycle graph with the specified number of nodes (6), representing
# the connectivity of a single hexagon. Nodes are integers 0 to 5.
H_op, node_map, num_sites = compute_H_sparse_heisenberg_graph(graph=hexagon_graph, J=J_val)
# Calls the function to compute the Hamiltonian operator for the generated graph.
# Returns the Hamiltonian as a SparsePauliOp, the node-to-qubit index map, and the number of sites.

# Basic validation
if num_sites != num_sites_hexagon:
    print(f"Error: Hexagon graph unexpectedly has {num_sites} sites instead of {num_sites_hexagon}.")
    exit()
if not (0 <= initial_excited_site_idx < num_sites and 0 <= plot_site_idx < num_sites):
     print(f"Error: Specified site indices ({initial_excited_site_idx}, {plot_site_idx}) are out of bounds [0, {num_sites-1}].")
     exit()

print(f"\nSystem has {num_sites} sites (qubits).")
# Prints the number of sites (qubits) in the simulation.
print(f"Initial state: Qubit {initial_excited_site_idx} in |1> (spin down), others in |0>.")
# Clarifies the initial state based on the parameter.
print(f"Plotting probability of Qubit {plot_site_idx} being in |1> (spin down).")
# Clarifies which qubit's spin down probability will be plotted.


print("\nConverting operators to dense matrices for exact classical simulation...")
# Status message before converting operators to matrices.
try:
# Starts a try block, as matrix conversion can fail for large systems (MemoryError).
    H_matrix = H_op.to_matrix(sparse=False)
    # Converts the Hamiltonian SparsePauliOp into a dense numpy matrix (2^N x 2^N).
    print(f"Hamiltonian matrix shape: {H_matrix.shape}")
    # Prints the shape of the Hamiltonian matrix.
    n_k_op = generate_single_number_operator_op(num_sites, plot_site_idx)
    # Generates the number operator (probability of being in |1>) for the specified plot_site_idx.
    print(f"Generated number operator SparsePauliOp for site {plot_site_idx}: {n_k_op}")
    # Prints the string representation of the generated number operator SparsePauliOp.
    n_k_matrix = n_k_op.to_matrix(sparse=False)
    # Converts the number operator SparsePauliOp into a dense numpy matrix (2^N x 2^N).
    print(f"Number operator matrix shape: {n_k_matrix.shape}")
    # Prints the shape of the number operator matrix.
    dim = 2**num_sites # Dimension of the Hilbert space
    # Calculates the dimension of the Hilbert space.
    # Basic check to ensure matrix dimensions match the Hilbert space dimension.
    if H_matrix.shape != (dim, dim) or n_k_matrix.shape != (dim, dim):
         raise ValueError("Matrix dimensions do not match expected Hilbert space dimension.")
except MemoryError:
    print(f"\nError: Out of memory. Cannot store {num_sites}-qubit matrices.")
    exit() # Exit if classical matrices are too large for memory
except Exception as e:
    print(f"\nError converting operators to matrices: {e}");
    exit() # Exit for other conversion errors

# --- Initial State Vector ---
# Sets up the initial state vector |psi(0)>.
# The initial state is |0...010...0> with '1' at the position of initial_excited_site_idx.
# In Qiskit's little-endian label convention |q_N-1 ... q_0>, the '1' corresponding
# to site_index k is at string index N-1-k.
initial_state_label_chars = ['0'] * num_sites
# Starts with a list of characters representing the all-|0> state.
initial_state_label_chars[num_sites - 1 - initial_excited_site_idx] = '1'
# Sets the character at the correct Qiskit label index to '1' for the excited site.
initial_state_label = "".join(initial_state_label_chars)
# Joins the characters to form the full Qiskit state label string.

try:
# Starts a try block for creating the initial state vector.
    psi_0_sv = Statevector.from_label(initial_state_label)
    # Creates the Qiskit Statevector object from the generated label string.
    psi_0 = psi_0_sv.data
    # Extracts the underlying dense numpy array from the Statevector. This is the
    # 2^N dimensional complex vector representing the initial state.
    # Prints the generated label and confirms which site it corresponds to.
    print(f"\nInitial state vector: Generated from label '|q_{num_sites-1}...q_0> = {initial_state_label}' (Qubit {initial_excited_site_idx} is spin down).")
    # Basic shape validation for the initial state vector.
    if psi_0.shape != (2**num_sites,):
         raise ValueError("Initial state vector shape incorrect.")
except Exception as e:
    print(f"Error creating initial state vector from label '{initial_state_label}': {e}");
    exit() # Exit if initial state creation fails


# --- Time Evolution Calculation ---
# This section calculates the state vector and observable expectation value at each time point.
print(f"\nCalculating exact time evolution up to t={t_max} for {num_time_points} points...")
# Status message.
occupation_prob_vs_time = np.zeros(len(times), dtype=np.float64)
# Initializes a numpy array to store the calculated occupation probabilities. Using float64 for precision.

for i, t in enumerate(tqdm(times, desc="Calculating time steps")):
# Loops through each time point with a progress bar.
    try:
    # Starts a try block for calculation at each time step.
        # U(t) = exp(-iHt) - Matrix exponential calculation
        U_t = expm(-1j * H_matrix * t) # H_matrix is already complex if needed after to_matrix
        # Calculates the exact time evolution operator U(t) using SciPy's expm function.
        # - -1j: The imaginary unit times -1.
        # - H_matrix: The dense Hamiltonian matrix.
        # - t: The current time point.
        # Matrix multiplication: |psi(t)> = U(t) |psi(0)>
        psi_t = U_t @ psi_0
        # Applies the calculated evolution operator U_t to the initial state vector psi_0
        # using matrix multiplication (@ operator) to get the state vector psi(t) at time t.

        # Calculate expectation value of n_k: <psi(t)| n_k |psi(t)>
        exp_val_nk = np.vdot(psi_t, n_k_matrix @ psi_t)
        # Calculates the expectation value of the number operator n_k in the state psi(t).
        # - n_k_matrix @ psi_t: Calculates the action of the operator on the state: n_k |psi(t)>.
        # - np.vdot(psi_t, ...): Calculates the inner product <psi(t)| (n_k |psi(t)>), which is
        #   the expectation value <psi(t)| n_k |psi(t)>.
        prob = np.real(exp_val_nk)
        # Extracts the real part of the expectation value (which represents the probability),
        # discarding any small imaginary part due to floating-point inaccuracies.
        occupation_prob_vs_time[i] = max(0.0, min(1.0, prob))
        # Stores the calculated probability for the current time step. Clamps the result
        # to the valid [0, 1] range to handle minor numerical errors.

        if i == 0:
        # Special check at the initial time point t=0.
            print(f"\nCHECK T=0: Calculated spin down probability for site {plot_site_idx} = {occupation_prob_vs_time[0]:.6f}")
            # Prints the calculated probability at t=0 for the monitored site.
            # Expected probability at t=0: 1 if the monitored site is the initially excited one, 0 otherwise.
            expected_p0 = 1.0 if initial_excited_site_idx == plot_site_idx else 0.0
            # Determines the theoretically exact probability at t=0.
            if not np.isclose(occupation_prob_vs_time[0], expected_p0, atol=1e-9):
            # Compares the calculated probability at t=0 to the expected value using a tolerance.
                 print(f"WARNING: Calculated probability at t=0 ({occupation_prob_vs_time[0]:.6f}) is not the expected {expected_p0} (within tolerance)!")
                 # Prints a warning if the check fails.
            else: print(f"Check PASSED: t=0 probability is {expected_p0} (within tolerance).")
            # Prints a success message if the check passes.
    except Exception as e:
    # Catches any error during the calculation for a specific time step.
        print(f"\nError during calculation at time step {i} (t={t:.2f}): {e}");
        # Prints an error message.
        occupation_prob_vs_time[i:] = np.nan; # Set remaining probabilities to NaN
        # Sets the current and all subsequent entries in the result array to NaN.
        break # Stop calculation loop early
        # Breaks out of the time evolution loop.
print("Calculation complete.")
# Prints a message indicating the end of the calculation loop.

# --- Smoothing Data ---
# Applies smoothing to the calculated data using a spline fit.
print("\nSmoothing the calculated probabilities...")
# Status message.
probs_smooth = None; times_smooth = None; smoothing_factor = 0.001
# Initializes variables for storing smoothed data and sets the smoothing factor.
try:
# Starts a try block for the smoothing process.
    valid_indices_smooth = ~np.isnan(occupation_prob_vs_time)
    # Creates a boolean mask to identify indices where the probability is not NaN.
    times_valid = times[valid_indices_smooth]; probs_valid = occupation_prob_vs_time[valid_indices_smooth]
    # Filters the time and probability arrays to include only valid data points.
    if len(times_valid) > 3: # Need at least ~k+1 points for spline degree k
    # Checks if there are enough valid points to perform spline smoothing (requires more than the spline degree).
        # Create a cubic spline (k=3)
        spline = UnivariateSpline(times_valid, probs_valid, s=smoothing_factor, k=3)
        # Creates a cubic spline object that fits the valid data points, controlled by 's'.
        times_smooth = np.linspace(times_valid.min(), times_valid.max(), 500)
        # Creates a denser array of time points spanning the range of valid data.
        probs_smooth = spline(times_smooth); # Evaluate the spline on the denser time grid
        probs_smooth = np.clip(probs_smooth, 0, 1) # Ensure smoothed probs are within [0, 1]
        # Evaluates the spline at the denser time points to get the smoothed probabilities. Clips the result.
        print(f"Smoothing successful (s={smoothing_factor}).")
        # Success message.
    else: print("Not enough valid points for smoothing (need > 3). Skipping smoothing.")
    # Message if smoothing is skipped.
except Exception as e:
    print(f"Warning: Smoothing failed - {e}.")
    # Prints a warning if smoothing encounters an error.


# --- Plotting Occupation Probability ---
# Generates the plot showing the spin down probability vs. time.
print(f"\nPlotting spin down probability for site index {plot_site_idx}...")
# Status message.
plt.figure(figsize=(10, 6)) # Create a new figure

# Filter out NaN values for plotting the raw data points
times_plot = times[~np.isnan(occupation_prob_vs_time)]
probs_plot = occupation_prob_vs_time[~np.isnan(occupation_prob_vs_time)]

if probs_smooth is not None and times_smooth is not None:
# If smoothing was successful, plot the smoothed curve and the raw points.
    plt.plot(times_smooth, probs_smooth, '-', color='black', linewidth=2, label='Exact (Smoothed)')
    # Plots the smoothed probability curve as a black line.
    plt.plot(times_plot, probs_plot, 'o', color='black', markersize=4, alpha=0.4, label='Exact (Raw Data)')
    # Plots the original calculated data points as markers.
else:
     # If smoothing failed, just plot the original raw data points connected by lines.
     plt.plot(times_plot, probs_plot, marker='.', linestyle='-', color='dodgerblue', label=f"Exact (Classical)")
     # Plots the valid calculated data points.

plt.xlabel("Time (units of 1/J)")
# Sets the label for the x-axis.
plt.ylabel(f"Probability (Spin Down)")
# Sets the label for the y-axis, clarifying it's the spin down probability.
# Set the title of the plot to reflect the exact simulation and plotted site/state.
title_str = (f"Exact Classical Simulation: Spin Down Probability of Site {plot_site_idx} vs. Time\n"
             f"(Initial state: Qubit {initial_excited_site_idx} spin down)")
plt.title(title_str)
# Sets the plot title, including details about the simulation type, plotted site, and initial state.
plt.xlim([times[0], times[-1]]); plt.ylim([-0.05, 1.05]) # Set axis limits
# Sets the x and y axis limits.
plt.legend(loc='best'); plt.grid(True); plt.tight_layout(); plt.show()
# Adds the legend, grid, adjusts layout, and displays the plot.

# --- Optional: Draw Lattice ---
# Draws the hexagonal lattice graph with highlighting for the initial and plotted sites.
draw_lattice = True # Set to False to skip drawing the lattice
if draw_lattice:
# Checks the flag to see if lattice drawing is enabled.
    try:
    # Starts a try block for drawing the lattice.
        print("\nDrawing Hexagon Lattice (N=6)...")
        # Status message.
        graph_vis = nx.cycle_graph(num_sites_hexagon) # Create graph for visualization
        # Recreates the cycle graph (hexagon).
        pos = nx.circular_layout(graph_vis) # Use circular layout for hexagon shape
        # Calculates node positions using a circular layout.
        labels = {i: str(i) for i in graph_vis.nodes()} # Node labels are qubit indices
        # Creates a dictionary mapping node indices to string labels for drawing.
        # Highlight nodes corresponding to initial excited site and plotted site.
        node_colors = ['lightblue'] * num_sites # Default node color
        # Initializes a list of colors for all nodes.
        if initial_excited_site_idx == plot_site_idx:
        # If the initial and plotted sites are the same.
             node_colors[initial_excited_site_idx] = 'royalblue' # Highlight that site in blue
             # Highlights the single relevant site in blue.
        else:
             # If the sites are different, highlight both.
             node_colors[initial_excited_site_idx] = 'orange' # Highlight initial site in orange
             node_colors[plot_site_idx] = 'cyan'       # Highlight plotted site in cyan
        nx.draw(graph_vis, pos, with_labels=True, labels=labels, node_size=600, node_color=node_colors, font_size=10, edge_color='gray')
        # Draws the graph using matplotlib. Sets colors based on the highlighting logic.
        plt.title("Hexagon Lattice - Nodes Labeled by Qubit Index") # Title for the lattice plot
        plt.gca().set_aspect('equal', adjustable='box'); # Ensure equal aspect ratio
        plt.show() # Display the lattice plot
    except Exception as e: print(f"Could not draw lattice: {e}")
    # Catches errors during lattice drawing and prints a message.
# --- End of Script ---

In [ ]:
# ==============================================================
#Heisenberg simulation using Qiskit Aer Simulator
# ==============================================================
# Describes the purpose of the script: simulating the time evolution of a
# Heisenberg model on a hexagonal lattice tile using Qiskit's Aer Simulator.

import numpy as np
# Imports the numpy library, used for numerical operations and array manipulation.

from qiskit import QuantumCircuit, transpile
# Imports QuantumCircuit for creating circuits and transpile for optimizing them.

from qiskit_aer import AerSimulator # Import AerSimulator
# Imports the AerSimulator class, which is a local simulator backend for execution.

from qiskit.quantum_info import SparsePauliOp
# Imports SparsePauliOp, used to represent the Hamiltonian efficiently.

from qiskit.circuit.library import PauliEvolutionGate
# Imports PauliEvolutionGate, a gate representing time evolution e^(-iHt).

from qiskit.synthesis import SuzukiTrotter
# Imports SuzukiTrotter, used to synthesize the PauliEvolutionGate into standard gates.

from qiskit.visualization import plot_histogram
# Imports plot_histogram for plotting measurement results (though not used in the final plot).

import networkx as nx
# Imports networkx, used for generating the hexagonal lattice graph.

import matplotlib.pyplot as plt
# Imports matplotlib.pyplot, used for creating the plot.

from tqdm import tqdm # For progress bar
# Imports tqdm for displaying progress bars around loops.

# --- HexagonTile Class Definition (ensure this is executed) ---
# Indicates the start of the class definition.
class HexagonTile:
# Defines the HexagonTile class, managing the lattice, Hamiltonian, and circuit building.
    def __init__(self, rows, cols, J=1.0, periodic=False):
    # Constructor method for the class.
        """
        Initialize a hexagonal lattice with rows and cols.
        :param rows: Number of rows in the lattice.
        :param cols: Number of columns in the lattice.
        :param J: Coupling constant for the Heisenberg interactions.
        :param periodic: If True, use periodic boundary conditions.
        """
        # Docstring explaining the constructor.
        if rows <= 0 or cols <= 0:
        # Validates input dimensions.
            raise ValueError("Rows and columns must be positive integers.")
            # Raises an error for invalid dimensions.

        # Generate the hexagonal lattice using networkx
        self.graph = nx.hexagonal_lattice_graph(rows, cols, periodic=periodic)
        # Generates the networkx graph for the lattice.
        self.qubit_count = len(self.graph.nodes)
        # Gets the number of sites (qubits) from the graph.
        self.edges = list(self.graph.edges)
        # Stores the list of edges.
        self.J = J
        # Stores the coupling constant J.
        self._hamiltonian = None # Cache Hamiltonian
        # Initializes a variable to cache the Hamiltonian.

        # Map node tuples to integer indices
        self.node_to_index = {node: i for i, node in enumerate(self.graph.nodes)}
        # Creates a mapping from networkx node identifiers to integer qubit indices.

        # Initialize qc here, but it will be overwritten or rebuilt
        self.qc = QuantumCircuit(self.qubit_count, name=f"Hex_{rows}x{cols}")
        # Creates an initial QuantumCircuit instance (not used in the main loop).

    def get_clean_circuit(self):
    # Method to create and return a new, empty quantum circuit.
        """Returns a new, empty QuantumCircuit for this tile."""
        return QuantumCircuit(self.qubit_count, name=f"Hex_{self.qubit_count}Q")
        # Creates and returns a QuantumCircuit with the correct number of qubits.

    def heisenberg_hamiltonian(self):
    # Method to construct the Heisenberg Hamiltonian as a SparsePauliOp.
        """Construct the Heisenberg Hamiltonian (XX + YY + ZZ). Caches result."""
        # Docstring.
        if self._hamiltonian is not None:
        # Checks if the Hamiltonian is cached.
            return self._hamiltonian
            # Returns cached Hamiltonian if available.

        pauli_list = []
        coeffs_list = []
        identity_list = list("I" * self.qubit_count)
        # Initializes lists to build the SparsePauliOp.

        for (u, v) in self.edges:
        # Iterates through each edge in the graph.
            i = self.node_to_index[u]
            j = self.node_to_index[v]
            # Gets qubit indices for the nodes of the edge.

            # Construct Pauli strings for XX, YY, ZZ terms on qubits i and j
            # The code constructs strings where characters at indices i and j correspond to qubits i and j.
            xx_list = identity_list[:]
            xx_list[i] = "X"; xx_list[j] = "X"
            # Creates the Pauli string for the XX term.
            pauli_list.append("".join(xx_list))
            coeffs_list.append(self.J)
            # Adds the XX term.

            yy_list = identity_list[:]
            yy_list[i] = "Y"; yy_list[j] = "Y"
            # Creates the Pauli string for the YY term.
            pauli_list.append("".join(yy_list))
            coeffs_list.append(self.J)
            # Adds the YY term.

            zz_list = identity_list[:]
            zz_list[i] = "Z"; zz_list[j] = "Z"
            # Creates the Pauli string for the ZZ term.
            pauli_list.append("".join(zz_list))
            coeffs_list.append(self.J)
            # Adds the ZZ term.

        if not pauli_list:
        # Handles case with no edges.
             self._hamiltonian = SparsePauliOp("I" * self.qubit_count, coeffs=[0.0])
             # Sets Hamiltonian to zero.
        else:
             self._hamiltonian = SparsePauliOp(pauli_list, coeffs=coeffs_list)
             # Creates the SparsePauliOp.

        return self._hamiltonian.simplify()
        # Returns the constructed and simplified Hamiltonian.

    def prepare_initial_state(self, circuit, excited_qubit=0):
    # Method to prepare the initial state on a circuit (|1> on a specified qubit).
        """Prepare the initial state (e.g., one excited qubit) on a given circuit."""
        # Docstring.
        if excited_qubit < 0 or excited_qubit >= self.qubit_count:
        # Validates the qubit index.
            raise ValueError("Invalid qubit index for excitation.")
            # Raises an error.
        circuit.reset(range(self.qubit_count)) # Ensure clean state
        # Adds reset gates to all qubits (sets them to |0>).
        circuit.x(excited_qubit)
        # Applies an X gate to the specified qubit (flips it to |1>).
        circuit.barrier() # Optional barrier for visual separation
        # Adds a barrier gate.
        return circuit
        # Returns the modified circuit.

    def simulate_time_evolution(self, circuit, time, trotter_steps=1, order=1):
    # Method to add time evolution gates to a circuit using Trotterization.
        """Add time evolution under the Heisenberg Hamiltonian to a given circuit."""
        # Docstring.
        if time <= 1e-9: return circuit # No evolution for zero time
        # Skips evolution if time is zero.
        if trotter_steps < 1: raise ValueError("Trotter steps must be >= 1")
        # Validates Trotter steps count.

        H = self.heisenberg_hamiltonian()
        # Gets the Hamiltonian.

        # Creates a PauliEvolutionGate representing e^(-iHt) with Trotter synthesis.
        evolution_gate = PauliEvolutionGate(H, time, synthesis=SuzukiTrotter(order=order, reps=trotter_steps))
        circuit.append(evolution_gate, range(self.qubit_count))
        # Appends the evolution gate to all qubits.
        circuit.barrier() # Optional barrier
        # Adds a barrier.
        return circuit
        # Returns the modified circuit.

    def add_measurements(self, circuit):
    # Method to add measurement gates to all qubits.
        """Add measurement gates to all qubits on a given circuit."""
        # Docstring.
        circuit.measure_all(inplace=True) # Adds measurements and classical bits
        # Uses measure_all to add measurements and classical registers.
        return circuit
        # Returns the modified circuit.

    def optimize_circuit(self, circuit, backend=None, optimization_level=1):
    # Method to transpile/optimize a circuit for a backend.
        """Optimize the circuit for reduced errors."""
        # Docstring.
        if backend is None:
        # Defaults backend to AerSimulator if none provided.
            backend = AerSimulator()
        optimized_qc = transpile(circuit, backend=backend, optimization_level=optimization_level)
        # Transpiles the circuit.
        return optimized_qc
        # Returns the transpiled circuit.

    def run_experiment(self, circuit, shots=15000, backend=None, optimization_level=1):
    # Method to run a circuit on a backend and get measurement counts.
        """Run the experiment for a given circuit and return measurement results."""
        # Docstring.
        if backend is None:
        # Defaults backend to AerSimulator.
            backend = AerSimulator()

        # Make sure measurements are added if not already
        has_measure = any(instr.operation.name == 'measure' for instr in circuit.data)
        # Checks if the circuit has measure instructions.
        if not has_measure and not circuit.cregs:
        # Adds measurements if missing.
             print("Warning: Adding measurements to the circuit before running.")
             self.add_measurements(circuit)

        optimized_qc = self.optimize_circuit(circuit, backend=backend, optimization_level=optimization_level)
        # Transpiles the circuit.

        job = backend.run(optimized_qc, shots=shots)
        # Runs the transpiled circuit.
        result = job.result()
        # Gets the job result.
        counts = result.get_counts(optimized_qc) # Get counts for the specific circuit
        # Extracts measurement counts.
        return counts
        # Returns the counts dictionary.

    def draw_lattice(self):
    # Method to draw the hexagonal lattice graph.
        """Draw the hexagonal lattice."""
        # Docstring.
        try:
        # Attempts to get node positions or use spring layout.
            pos = nx.get_node_attributes(self.graph, 'pos')
            if not pos: pos = nx.spring_layout(self.graph)
        except KeyError:
        # Uses spring layout if positions are not found.
             pos = nx.spring_layout(self.graph)
        nx.draw(self.graph, pos, with_labels=True, node_size=500, node_color='lightblue', font_weight='bold')
        # Draws the graph.
        plt.title(f"Hexagonal Lattice ({len(self.graph.nodes)} sites)")
        # Sets the plot title.
        plt.show()
        # Displays the plot.
# --- End of HexagonTile Class Definition ---


# --- Simulation Parameters ---
# Defines parameters for the simulation run.
rows, cols = 1, 1
# Sets lattice dimensions (e.g., 1x1 hex graph usually has 6 sites).
J = 1.0
# Sets the coupling constant.
initial_excited_qubit = 0
# Sets the index of the qubit initially in the |1> (spin down) state.
plot_site_index = 0
# Sets the index of the qubit whose spin down probability is plotted.
trotter_steps_for_sim = 15
# Sets the number of Trotter steps for time evolution.
shots_for_sim = 15000
# Sets the number of measurement shots per circuit.
lattice_periodic = False
# Sets boundary conditions for the lattice graph.

# Define time points
times = np.linspace(0, 6.25, 100)
# Creates an array of time points for the simulation.

# --- Initialize ---
# Sets up the tile object and simulator backend.
tile_setup = HexagonTile(rows=rows, cols=cols, J=J, periodic=lattice_periodic)
# Creates a HexagonTile instance.
num_qubits = tile_setup.qubit_count
# Gets the number of qubits.
probabilities = [] # List to store P(site=1) at each time step
# Initializes list to store calculated probabilities.
backend_sim = AerSimulator() # Create simulator instance once
# Creates an AerSimulator instance.

print(f"Simulating dynamics for site {plot_site_index} on a {rows}x{cols} lattice ({num_qubits} qubits).")
# Prints simulation details.
print(f"Initial state: Qubit {initial_excited_qubit} in |1> (spin down)")
# Prints initial state configuration.
print(f"Trotter steps: {trotter_steps_for_sim}, Shots: {shots_for_sim}")
# Prints Trotter steps and shots.

# --- Time Evolution Loop ---
# Main loop simulating evolution and processing results for each time point.
for t in tqdm(times):
# Loops through time points with a progress bar.
    # 1. Create a clean circuit for this time step
    qc = tile_setup.get_clean_circuit()
    # Creates a new circuit.

    # 2. Prepare initial state
    tile_setup.prepare_initial_state(qc, excited_qubit=initial_excited_qubit)
    # Prepares the initial state on the circuit.

    # 3. Simulate time evolution
    tile_setup.simulate_time_evolution(qc, time=t, trotter_steps=trotter_steps_for_sim)
    # Adds time evolution gates.

    # 4. Add measurements (handled in run_experiment if needed)

    # 5. Run the experiment
    counts = tile_setup.run_experiment(qc, shots=shots_for_sim, backend=backend_sim, optimization_level=1)
    # Runs the circuit on the simulator and gets counts.

    # 6. Calculate probability for the target site
    # Calculates the probability of the target qubit being in the |1> state from counts.
    excitation_counts = 0
    # Initializes count for |1> outcomes on the target qubit.
    total_counts = sum(counts.values())
    # Gets total shot counts.
    num_qubits = tile_setup.qubit_count
    # Gets number of qubits.

    for bitstring, count in counts.items():
    # Iterates through measurement outcomes and counts.
        # Check the bit for plot_site_index in the little-endian string
        # Bitstring index for qubit j is num_qubits - 1 - j
        index_to_check = num_qubits - 1 - plot_site_index
        # Calculates the bitstring index for the target qubit (little-endian).
        if len(bitstring) == num_qubits and bitstring[index_to_check] == '1':
        # Checks if bitstring length is correct and the bit for the target qubit is '1'.
             excitation_counts += count
             # Adds count if target qubit was '1'.

    probability_site_1 = excitation_counts / total_counts if total_counts > 0 else 0.0
    # Calculates the probability.
    probabilities.append(probability_site_1)
    # Appends probability to the list.

# --- Plotting ---
# Generates and displays the final plot.
plt.figure(figsize=(8, 5))
# Creates a figure.
plt.plot(times, probabilities, marker='o', linestyle='-', label=f'N={trotter_steps_for_sim}', color='orange')
# Plots the time vs. probability data.
plt.xlabel("Time (units of 1/J)")
# Sets x-axis label.
plt.ylabel(f"Probability (Spin Down)")
# Sets y-axis label, clarifying 'spin down'.
plt.title(f"Simulator: Spin Down Probability of Site {plot_site_index} vs. Time")
# Sets the plot title.
plt.ylim([-0.05, 1.05])
# Sets y-axis limits.
plt.grid(True)
# Adds grid.
plt.legend()
# Displays legend.
plt.tight_layout()
# Adjusts layout.
plt.show()
# Displays the plot
#--------end of script------------

In [ ]:
# ==============================================================
#  Hardware Heisenberg simulation with LOCAL read‑out mitigation
# ==============================================================
#  the overall purpose of the script:
# Simulating the time evolution of a Heisenberg model on a hexagonal lattice tile,
# running the simulation on actual quantum hardware via Qiskit Runtime (SamplerV2),
# and applying local readout error mitigation using the 'mthree' library.

import os
# Imports the 'os' module, providing a way to interact with the operating system,
# commonly used here to access environment variables like the IBM Quantum token.

import numpy as np
# Imports the 'numpy' library, essential for numerical operations, especially
# array and matrix manipulation, used for defining time points, probabilities, etc.

import networkx as nx
# Imports the 'networkx' library, used for creating and manipulating graph structures.
# Here, it's specifically used to generate the hexagonal lattice graph.

import matplotlib.pyplot as plt
# Imports the 'matplotlib.pyplot' module, used for creating plots and visualizations,
# specifically for plotting the probability vs. time curve.

from collections import Counter
# Imports the 'Counter' class from the 'collections' module. While imported,
# it's not explicitly used in the main execution path for processing Sampler results,
# as SamplerV2 provides counts directly. 

from tqdm import tqdm
# Imports the 'tqdm' library, used to display progress bars, which are useful for
# tracking the progress of loops (like building circuits or processing results)
# especially when running on potentially slow hardware.

import sys # Import sys to potentially exit on critical errors
# Imports the 'sys' module, providing access to system-specific parameters and functions.
# Here, it's used to exit the script gracefully if critical errors occur (e.g., missing token,
# failed backend connection, failed calibration).

from qiskit import QuantumCircuit, transpile
# Imports core Qiskit classes:
# - QuantumCircuit: Represents the quantum circuit itself, a sequence of quantum gates.
# - transpile: A function to optimize and map circuits for a specific quantum backend.

from qiskit.quantum_info import SparsePauliOp
# Imports 'SparsePauliOp' from 'qiskit.quantum_info'. This class is used to represent
# a Hamiltonian as a sum of Pauli operators with coefficients, in a memory-efficient sparse format.

from qiskit.circuit.library import PauliEvolutionGate
# Imports 'PauliEvolutionGate' from 'qiskit.circuit.library'. This is a standard gate
# that represents the time evolution $e^{-iHt}$ for a Hamiltonian H which is a SparsePauliOp.
# It handles the decomposition into implementable gates using a synthesis method like Trotter.

from qiskit.synthesis import SuzukiTrotter
# Imports 'SuzukiTrotter' from 'qiskit.synthesis'. This class provides a method to
# decompose the 'PauliEvolutionGate' into a sequence of simpler gates using the
# Suzuki-Trotter approximation formula.

from qiskit_ibm_runtime import QiskitRuntimeService, Session, SamplerV2 as Sampler
# Imports necessary components for using the Qiskit Runtime service and primitives:
# - QiskitRuntimeService: Used to connect to and manage access to IBM Quantum systems.
# - Session: Manages a persistent connection to a backend for submitting multiple jobs within the same context.
# - SamplerV2: A Qiskit Runtime primitive that takes a quantum circuit and returns
#   a dictionary of measurement outcomes (bitstrings) with associated probabilities or counts. V2 is the newer version.

# Import Counts object type is usually not needed for simple access
# from qiskit.result import Counts

import mthree  # ← local measurement‑error mitigation
# Imports the 'mthree' library (Measurement Mitigation via Mutli-fidelity MEasurmenTs).
# This library is used specifically for correcting errors introduced during the
# measurement process on quantum hardware (readout errors).

# ------------------------------------------------------------------
#  Login & backend
# ------------------------------------------------------------------
# This section handles connecting to the IBM Quantum service and selecting a backend.

# Replace with your valid token or use environment variable QISKIT_IBM_TOKEN
# Using environment variable is generally safer than hardcoding
# token = os.environ.get("QISKIT_IBM_TOKEN")
# from an environment variable for security.

# If you must hardcode (for testing/example, not recommended in production):
token = "QISKIT_IBM_TOKEN"
# This line hardcodes the IBM Quantum API token. This is convenient for testing
# but is a security risk in production code. **Replace this with your actual token
# or uncomment the environment variable line.**

if not token:
# Checks if the 'token' variable is empty or None.
    print("Error: IBM Quantum token not found.")
    # Prints an error message if the token is missing.
    print("Set the QISKIT_IBM_TOKEN environment variable or replace the placeholder token.")
    # Provides instructions on how to fix the missing token.
    # Exit gracefully if token is missing
    sys.exit("IBM Quantum token not configured.")
    # Exits the script immediately with an error message if the token is not set.

try:
# Starts a try block to catch potential exceptions during connection and backend selection.
    service = QiskitRuntimeService(channel="ibm_quantum", token=token)
    # Initializes the QiskitRuntimeService object, connecting to the IBM Quantum
    # cloud service using the specified channel ("ibm_quantum") and the provided token.
    # Using 'CHOSEN_BACKEND'.
    backend = service.backend("CHOSEN_BACKEND")
    # Selects a specific quantum backend (quantum processor) by its name.
    # from the available backends through the initialized service.
except Exception as e:
# Catches any exception that occurs within the try block.
     print(f"Error connecting to IBM Quantum Experience or getting backend: {e}")
     # Prints an informative error message including the exception details.
     sys.exit("Failed to initialize IBM Quantum backend.")
     # Exits the script with an error message if connecting or getting the backend fails.

print("Using backend:", backend.name)
# Prints the name of the successfully selected backend to the console.

# ------------------------------------------------------------------
#  Honeycomb lattice construction
# ------------------------------------------------------------------
# This section defines a class and its methods to handle the specifics of
# the hexagonal lattice and Hamiltonian for the simulation.

class HexagonTile:
# Defines the HexagonTile class.
    def __init__(self, rows, cols, J=1.0, periodic=False):
    # The constructor method for the HexagonTile class.
    # Takes the number of rows and columns for the lattice, the coupling constant J,
    # and a flag for periodic boundary conditions.
        if rows <= 0 or cols <= 0:
        # Basic input validation for rows and cols.
            raise ValueError("rows and cols must be >0")
            # Raises an error if dimensions are invalid.

        # networkx hexagonal_lattice_graph nodes are tuples (j, i) where j is col, i is row.
        # Map these to qubit indices 0, 1, 2...
        # Note: (1,1) non-periodic yields a 6 qubits hexagon.
        # The code adapts to self.qubit_count derived from the graph.
        self.graph = nx.hexagonal_lattice_graph(rows, cols, periodic=periodic)
        # Generates the hexagonal lattice graph using networkx based on the input parameters.

        # Check if the graph is empty
        if not self.graph.nodes:
        # Checks if the generated graph has any nodes.
             raise ValueError(f"Created an empty graph with rows={rows}, cols={cols}, periodic={periodic}. Adjust dimensions.")
             # Raises an error if the graph is empty, as it won't represent a physical system.

        self.nodes_list = list(self.graph.nodes) # Ensure a consistent order
        # Stores a list of the graph nodes. Using a list ensures a fixed order,
        # which is important for mapping nodes to consistent qubit indices.
        self.qubit_count = len(self.nodes_list)
        # Gets the number of nodes (sites) in the graph, which is equal to the number of qubits.
        self.J = J
        # Stores the coupling constant J.
        self.edges = list(self.graph.edges)
        # Stores a list of edges (connections) in the graph.

        # Create a mapping from node tuple (j, i) to integer qubit index
        self.node_to_qubit = {node: i for i, node in enumerate(self.nodes_list)}
        # Creates a dictionary mapping each networkx node tuple (like (0,0), (0,1))
        # to a unique integer qubit index (0, 1, 2,...), based on the order in self.nodes_list.
        self.qubit_to_node = {i: node for node, i in self.node_to_qubit.items()}
        # Creates the inverse mapping, from qubit index back to the networkx node tuple.

        self._H = None
        # Initializes a variable to cache the Hamiltonian once it's computed,
        # so it doesn't need to be recalculated every time.

    def _hamiltonian(self):
    # Method to construct and return the Heisenberg Hamiltonian as a SparsePauliOp.
        if self._H is not None:
        # Checks if the Hamiltonian has already been computed and cached.
            return self._H
            # If cached, returns the cached Hamiltonian.

        paulis, coeffs = [], []
        # Initializes lists to store Pauli strings and their corresponding coefficients.
        I_string = "I" * self.qubit_count
        # Creates a base string of 'I' (Identity) operators, one for each qubit.

        for (u, v) in self.edges:
        # Iterates through each edge (u, v) in the graph.
            # Get qubit indices from node tuples
            i, j = self.node_to_qubit[u], self.node_to_qubit[v]
            # Gets the integer qubit indices corresponding to the nodes u and v of the current edge.
            for axis in "XYZ":
            # For each edge, iterate through the interaction types: XX, YY, ZZ.
                label_list = list(I_string)
                # Starts with a list of identities for all qubits.
                label_list[i] = axis
                # Sets the operator for qubit i to the current axis (X, Y, or Z).
                label_list[j] = axis
                # Sets the operator for qubit j to the current axis (X, Y, or Z).
                paulis.append("".join(label_list))
                # Joins the list back into a string (e.g., 'IIXIIYII') and adds it to the paulis list.
                coeffs.append(self.J)
                # Adds the coupling constant J as the coefficient for this Pauli term.

        # Create SparsePauliOp and simplify
        self._H = SparsePauliOp(paulis, coeffs).simplify()
        # Creates a SparsePauliOp from the collected Pauli strings and coefficients.
        # .simplify() combines identical Pauli terms if any were generated.
        return self._H
        # Returns the constructed (and now cached) Hamiltonian.

    # ---------- convenience wrappers ----------
    # This section groups methods that help build the quantum circuit steps.

    def bare_circuit(self):
    # Method to create a new, empty quantum circuit for the lattice size.
        # Circuits for SamplerV2 must have a classical output measure
        # measure_all will handle creating the classical register
        return QuantumCircuit(self.qubit_count, name="HexTile")
        # Creates a QuantumCircuit with the number of qubits determined by the lattice size.
        # Gives it a name. Note that SamplerV2 requires measurements, which will be added later.

    def prepare_source(self, qc, q0_qubit_index=0):
    # Method to prepare a specific initial state on a given circuit 'qc'.
    # By default, it prepares the state where qubit 0 is |1> (spin down) and others are |0>.
        """Prepares the initial state with an excitation at the given qubit index."""
        if not 0 <= q0_qubit_index < self.qubit_count:
        # Validates the input qubit index.
             raise ValueError(f"Initial qubit index {q0_qubit_index} out of range [0, {self.qubit_count-1}]")
             # Raises an error if the index is invalid.
        # qc.reset(range(self.qubit_count)) # Reset is often implicit but good practice
        # qubits start in the |0> state, especially on real hardware.
        qc.x(q0_qubit_index) # Apply X to get |1> state (excitation) - corresponds to spin-down
        # Applies an X gate (quantum NOT) to the specified qubit. Since qubits
        # are typically initialized to |0>, this flips the target qubit to |1>.
        # In this spin model context, |1> corresponds to "spin down".
        qc.barrier()
        # Inserts a barrier gate for visualization and sometimes preventing optimizations
        # across circuit sections.
        return qc
        # Returns the modified circuit.

    def time_evolve(self, qc, t, trotter=8, order=1):
    # Method to add the time evolution of the Hamiltonian for time 't' to circuit 'qc'.
        if t <= 0:
        # Checks if the evolution time is zero or negative.
            # If time is zero or negative, just return the circuit without evolution
            return qc
            # If time is not positive, returns the circuit unchanged.

        # Ensure Hamiltonian is built
        hamiltonian = self._hamiltonian()
        # Calls the private method to get the Hamiltonian (computes it if not cached).

        # PauliEvolutionGate directly accepts SparsePauliOp
        # Creates a PauliEvolutionGate representing e^{-i * hamiltonian * t}.
        # It uses the specified Suzuki-Trotter synthesis method with the given order and reps (trotter steps).
        evo = PauliEvolutionGate(hamiltonian, t,
                                 synthesis=SuzukiTrotter(order, reps=trotter))
        qc.append(evo, qc.qubits)
        # Appends the time evolution gate to the circuit, applying it to all qubits.
        qc.barrier()
        # Inserts a barrier after the evolution.
        return qc
        # Returns the modified circuit.

    @staticmethod
    def measure_all(qc):
    # Static method to add measurement gates to all qubits in a circuit.
        """
        Measures all qubits and automatically adds measurements.
        """
        # Use Qiskit's measure_all with inplace=True to add measurements directly
        qc.measure_all(inplace=True)
        # This Qiskit method automatically adds a classical register (usually named 'meas'
        # or 'c') equal to the number of qubits and adds a measurement gate from each
        # qubit to the corresponding classical bit. 'inplace=True' modifies the circuit directly.
        # This is required for SamplerV2.
        return qc
        # Returns the modified circuit.


# ------------------------------------------------------------------
#  Simulation parameters
# ------------------------------------------------------------------
# This section sets the specific parameters for the simulation run.

# Using (1,1) tile as per your latest outputs showing 6 qubits
rows, cols = 1, 1
# Sets the dimensions of the hexagonal lattice section. (1,1) non-periodic
# typically results in 6 qubits in networkx.
periodic = False  # Explicitly set periodic for clarity
# Sets whether to use periodic boundary conditions for the lattice graph.
J = 1.0
# Sets the coupling constant for the Heisenberg interaction.
initial_qubit_index = 0  # Qubit index where initial excitation is placed (qubit 0)
# Specifies the integer index of the qubit that will be initialized in the |1> state.
plot_site_qubit_index = 0  # Qubit index whose probability of being |1> we plot (qubit 0)
# Specifies the integer index of the qubit for which we want to track and plot the
# probability of being in the |1> (spin-down) state.
trotter_steps = 3
# Sets the number of Trotter steps (reps) used in the Suzuki-Trotter synthesis.
# More steps generally mean better accuracy but a longer circuit.
times = np.linspace(0.0, 6.25, 100) # Reduced steps for faster test
# Creates a numpy array of time points for the simulation, from 0.0 to 6.25, with 100 points.
shots = 15000
# Sets the number of measurement shots to run for each circuit (at each time point).
# More shots reduce statistical noise.

try:
# Starts a try block to handle potential errors during HexagonTile creation.
    tile = HexagonTile(rows, cols, J, periodic=periodic)
    # Creates an instance of the HexagonTile class with the specified parameters.
except ValueError as e:
# Catches a ValueError (which the HexagonTile constructor might raise).
    print(f"Error creating HexagonTile: {e}")
    # Prints the error message.
    sys.exit("Failed to create HexagonTile.")
    # Exits the script if HexagonTile creation fails due to invalid parameters.

# --- Check qubit indices against the actual qubit count ---
# This section performs crucial validation after the HexagonTile object is created.
if tile.qubit_count == 0:
# Checks if the created tile resulted in zero qubits.
    print(f"HexagonTile created 0 qubits with rows={rows}, cols={cols}, periodic={periodic}.")
    print("Adjust rows/cols parameters to create a valid graph.")
    sys.exit("HexagonTile has 0 qubits.")
    # Prints error message and exits if no qubits were created (likely invalid graph params).

if not 0 <= initial_qubit_index < tile.qubit_count:
# Checks if the specified initial qubit index is within the valid range [0, qubit_count-1].
     print(f"Initial qubit index {initial_qubit_index} is out of range [0, {tile.qubit_count-1}] for {tile.qubit_count} qubits.")
     sys.exit("Invalid initial_qubit_index.")
     # Prints error and exits if the index is out of bounds.
if not 0 <= plot_site_qubit_index < tile.qubit_count:
# Checks if the specified plot site qubit index is within the valid range.
     print(f"Plot site qubit index {plot_site_qubit_index} is out of range [0, {tile.qubit_count-1}] for {tile.qubit_count} qubits.")
     sys.exit("Invalid plot_site_qubit_index.")
     # Prints error and exits if the index is out of bounds.


# --- print statement  ---
print(f"→ {tile.qubit_count} qubit(s) (Tile params: rows={rows}, cols={cols}, periodic={tile.graph.graph.get('periodic', False)}); monitoring qubit {plot_site_qubit_index}")
# Prints confirmation about the number of qubits created and which qubit is being monitored.
# It uses tile.qubit_count and plot_site_qubit_index for accuracy.
# The .get('periodic', False) safely retrieves the periodic flag from the graph properties.

# Optional: print qubit mapping to node (j, i)
print("Qubit to node mapping:")
# Prints a header for the mapping.
for i in range(tile.qubit_count):
# Loops through each qubit index.
    print(f"Qubit {i} maps to node {tile.qubit_to_node[i]}")
    # Prints the mapping from the integer qubit index back to the networkx node tuple.
    # This helps understand which site corresponds to which qubit index.


# ------------------------------------------------------------------
#  Build circuits
# ------------------------------------------------------------------
# This section generates the quantum circuits for each time point.

circs = []
# Initializes an empty list to store the generated quantum circuits.
for t in tqdm(times, desc="Building circuits"):
# Loops through each time point 't' in the 'times' array. Displays a progress bar using tqdm.
    qc = tile.bare_circuit()
    # Creates a new, empty QuantumCircuit for the current time step.
    # Pass the qubit index directly
    tile.prepare_source(qc, initial_qubit_index)
    # Prepares the initial state on the circuit 'qc', setting the specified qubit to |1>.
    tile.time_evolve(qc, t, trotter_steps)
    # Adds the time evolution gates (Trotterized) to the circuit 'qc' for the current time 't'.
    # Call HexagonTile.measure_all with the circuit object 'qc'
    tile.measure_all(qc) # 'inplace=True' is handled *inside* the method
    # Adds measurement gates to all qubits in the circuit 'qc'.
    circs.append(qc)
    # Adds the completed circuit for the current time step to the list 'circs'.

print(f"✓ Built {len(circs)} circuits.")
# Prints a confirmation message showing how many circuits were built.

# ------------------------------------------------------------------
#  Transpile circuits for the backend
# ------------------------------------------------------------------
# This section optimizes and maps the generated circuits for the chosen IBM Quantum backend.

print(f"Transpiling {len(circs)} circuits for {backend.name}...")
# Prints a status message before starting transpilation.
# Transpile the list of circuits
# Use a high optimization level to break down PauliEvolutionGate
try:
# Starts a try block to handle potential errors during transpilation.
    transpiled_circs = transpile(circs, backend=backend, optimization_level=3)
    # Calls Qiskit's transpile function.
    # - 'circs': The list of circuits to transpile.
    # - 'backend': The target backend hardware.
    # - 'optimization_level=3': Applies a high level of optimization, which is needed
    #   to break down the PauliEvolutionGate into the backend's native gates and
    #   optimize the circuit depth and gate count.
    print("✓ Circuits transpiled.")
    # Prints success message if transpilation completes without errors.
    # Optional: Print a transpiled circuit to inspect basis gates and classical register name
    # print("\nFirst Transpiled Circuit:")
    # print(transpiled_circs[0].draw(output='text'))
    # print("-" * 30)
except Exception as e:
# Catches any exception during transpilation.
     print(f"Error during transpilation: {e}")
     # Prints an error message.
     # If transpilation fails, running the circuits won't work
     sys.exit("Circuit transpilation failed.")
     # Exits the script if transpilation fails.

# ------------------------------------------------------------------
#  Run inside a session with mthree mitigation
# ------------------------------------------------------------------
# This is the main section where the circuits are executed on the quantum hardware
# via Qiskit Runtime and measurement errors are mitigated using mthree.

probabilities = [] # List to store probabilities for plotting
# Initializes an empty list to store the calculated probability (of the monitored
# qubit being |1>) for each time point.

# Define qubits for mitigation (all qubits used in the circuit)
qubits_to_mitigate = list(range(tile.qubit_count))
# Creates a list of all qubit indices [0, 1, ..., qubit_count-1]. mthree needs
# to know which qubits' measurement errors to calibrate and correct.

# Flag to print data_bin keys only for the first result
printed_databin_keys = False
# A flag used to control printing debugging information about the Sampler result structure
# only once for the first circuit result.

# 1) open ONE session for everything
try:
# Starts a try block for the entire runtime session and result processing.
    # Initialize Session with just the backend.
    # The session object holds the backend and service context.
    with Session(backend=backend) as sess:
    # Uses Qiskit Runtime's Session as a context manager. This opens a connection
    # to the backend that persists for the duration of the 'with' block, allowing
    # multiple jobs to potentially run faster and more efficiently. It automatically
    # closes the session upon exiting the block.

        # 2) Initialize mthree mitigation *inside* the session
        print("Starting mthree calibration...")
        # Status message for mthree calibration.
        mit = mthree.M3Mitigation(backend)
        # Initializes the mthree mitigation object, linking it to the selected backend.
        try:
        # Starts a try block for the mthree calibration step.
            mit.cals_from_system(
        # Runs the calibration circuits needed by mthree on the backend.
                qubits_to_mitigate,
                # Specifies the qubits for which calibration data should be collected.
                shots=15000, # Calibration shots
                # Sets the number of shots for the calibration circuits.
                runtime_mode="session",   # Piggy‑back on the active session
                # Tells mthree to submit calibration jobs within the currently active Runtime session.
                async_cal=False           # Process calibration batches sequentially
                # Determines if calibration batches run asynchronously. False means sequential execution.
            )
            print("✓ Obtained mthree calibration matrices.")
            # Prints success message if calibration completes.
        except Exception as e:
        # Catches any exception during mthree calibration.
             print(f"Error during mthree calibration: {e}")
             # Prints error message.
             # Continue without mitigation if calibration fails? Or exit? Let's exit for now.
             sys.exit("mthree calibration failed.")
             # Exits the script if calibration fails.

        # 3) build the Sampler *inside* the session
        # Initialize SamplerV2 *without* backend or service when in a session
        sampler = Sampler()
        # Initializes the SamplerV2 primitive object. When initialized inside a
        # Session context, it automatically inherits the backend and service from the session.

        # 4) Run the batch of circuits
        print(f"Running {len(transpiled_circs)} transpiled circuits on {backend.name} with {shots} shots each...")
        # Status message before submitting jobs.
        # Use the TRANSPILED circuits list here
        job = sampler.run(transpiled_circs, shots=shots)
        # Submits the list of *transpiled* circuits to the Sampler primitive for execution
        # on the backend, specifying the number of shots for each circuit. This returns a Job object.

        # 5) Wait for job completion and retrieve results
        # 6) Process results and apply mitigation
        print("Processing results and applying mitigation...")
        # Status message for result processing.

        result = job.result() # Get the PrimitiveResult object
        # Waits for the job to complete on the backend and retrieves the results.
        # 'result' is a PrimitiveResult object containing results for all circuits submitted in the batch.
        print("✓ Job completed successfully.")
        # Prints success message if the job completes without throwing an exception here.

        # --- Accessing counts based on standard V2 structure (pub_result -> data -> classical register name) ---
        # Iterates through the results for each circuit submitted in the batch.
        for i, pub_result in tqdm(enumerate(result), total=len(transpiled_circs), desc="Processing results"):
        # 'result' contains a list of SamplerPubResult objects, one for each circuit.
        # 'enumerate' provides the index 'i' (which circuit this is) and the result 'pub_result'.
        # tqdm wraps the loop to show a progress bar.
            # Access the DataBin from the SamplerPubResult
            data_bin = pub_result.data
            # The actual measurement data (counts, quasi-probabilities, etc.) for a single
            # circuit result is stored in the 'data' attribute, which is a DataBin object.

            # --- DEBUGGING: Print keys in the DataBin (only for the first result) ---
            if not printed_databin_keys:
            # Checks the flag to only print this debugging info once.
                print("\n--- Debugging DataBin Keys (First Result) ---")
                print(f"Type of data_bin: {type(data_bin)}")
                try:
                # Tries to list keys in the DataBin.
                    available_keys = list(data_bin.keys())
                    print(f"Available keys in data_bin: {available_keys}")
                    # Based on previous output, the key is 'meas'
                    # if 'meas' not in available_keys:
                    #      print("CRITICAL WARNING: 'meas' key not found. Sampler results structure may have changed.")
                    # This commented code provides hints on how to check for the expected key.
                except Exception as key_error_debug:
                # Catches errors if key access fails.
                    print(f"Could not list keys from data_bin: {key_error_debug}")
                print("--------------------------------------------\n")
                printed_databin_keys = True
                # Sets the flag so this block doesn't run again.
            # --- END DEBUGGING ---

            raw_counts = {} # Initialize empty counts for this circuit
            # Initializes an empty dictionary to hold the raw measurement counts for the current circuit.

            try:
            # Starts a try block for accessing and processing counts.
                # Based on debug output, the correct key is 'meas'.
                classical_register_key = 'meas'
                # Specifies the key expected in the DataBin to get the counts. This is
                # the name of the classical register used for measurement (set by measure_all).

                # Check if the key exists before accessing
                if classical_register_key not in data_bin:
                # Checks if the expected classical register key is actually present in the DataBin.
                     print(f"Error: Classical register key '{classical_register_key}' not found in data_bin for circuit {i}.")
                     print(f"Available keys were: {list(data_bin.keys())}")
                     # Prints error and available keys if the key is missing.
                     # If key is missing, skip this result
                     probabilities.append(np.nan) # Append NaN to keep time array aligned
                     # Appends 'Not a Number' to the probabilities list to signify that
                     # processing failed for this time point, maintaining list length.
                     continue # Skip the rest of the loop for this circuit result.

                # Get the counts. This returns a dictionary like {'hex_string': count}.
                raw_counts = data_bin[classical_register_key].get_counts()
                # Accesses the counts data using the specified key and calls .get_counts()
                # to get a dictionary mapping bitstring representations (usually hex) to integers counts.

            except Exception as e:
            # Catches any exception during count access.
                print(f"An error occurred accessing counts for circuit {i}: {e}")
                import traceback
                traceback.print_exc()
                # Prints error message and a full traceback for debugging.
                probabilities.append(np.nan) # Append NaN
                # Appends NaN to the probabilities list.
                continue
                # Skips the rest of the loop for this circuit result.


            # Apply mthree mitigation
            # Ensure raw_counts is not empty before applying mitigation
            if not raw_counts:
            # Checks if the raw counts dictionary is empty.
                 # This can happen if shots=0 or if something went wrong on the backend
                 print(f"Warning: No counts returned for circuit {i} at time {times[i]:.2f}. Skipping mitigation and probability calculation.")
                 # Prints a warning if no counts were obtained.
                 probabilities.append(np.nan) # Append NaN
                 # Appends NaN.
                 continue
                 # Skips the rest of the loop for this circuit result.

            try:
            # Starts a try block for applying mitigation and calculating probability.
                # Use mit.apply_correction instead of mit.apply
                # mit.apply_correction expects a dictionary of counts (which get_counts provides)
                mitigated_quasi_probs = mit.apply_correction(raw_counts, qubits=qubits_to_mitigate)
                # Applies the mthree measurement error correction to the raw counts.
                # It uses the calibration data obtained earlier for the specified qubits.
                # Returns a dictionary of *quasi-probabilities* (which can be negative or >1)
                # mapping bitstring representations to corrected probabilities.

                # Calculate probability of plot_site_qubit_index being in state |1>
                p_exc = 0.0
                # Initializes a variable to accumulate the quasi-probability of the target qubit being in the |1> state.
                # mitigated_quasi_probs is a dictionary mapping state strings (usually hex) to quasi-probabilities
                for state_hex_or_str, quasi_prob in mitigated_quasi_probs.items():
                # Iterates through each measurement outcome and its corrected quasi-probability.
                     try:
                         # Convert the hex string key to an integer
                         state_int = int(str(state_hex_or_str), 16)
                         # Converts the bitstring key (usually in hexadecimal format from Sampler)
                         # into an integer representation.
                     except ValueError:
                          print(f"Warning: Could not convert state key '{state_hex_or_str}' to integer for circuit {i}. Skipping this state.")
                          continue # Skip this state if key is unparseable
                          # Handles cases where the key isn't a valid hex string by skipping that outcome.

                     # Check if the bit corresponding to plot_site_qubit_index is set (is 1)
                     # SamplerV2 results are LITTLE-ENDIAN BIT ORDERING by default.
                     if ((state_int >> plot_site_qubit_index) & 1):
                     # This is the key step for calculating the probability of the target qubit being |1>.
                     # - '>> plot_site_qubit_index': Right-shifts the integer representation of the bitstring
                     #   so that the bit corresponding to the 'plot_site_qubit_index' qubit becomes the rightmost bit (bit 0).
                     #   SamplerV2 uses little-endian, so bit 0 is qubit 0, bit 1 is qubit 1, etc. This check is correct for little-endian.
                     # - '& 1': Performs a bitwise AND with 1. This checks if the rightmost bit (which is now the bit
                     #   for the target qubit after shifting) is 1.
                     # If the condition is True (the bit is 1), the measurement outcome corresponds to the target qubit being in the |1> state.
                         p_exc += quasi_prob
                         # If the bit is 1, adds the quasi-probability of this outcome to the sum 'p_exc'.

                # Append the calculated probability for the current circuit
                # probabilities from mthree can be outside [0, 1], clip them for plotting
                probabilities.append(np.clip(p_exc, 0.0, 1.0))
                # After summing up quasi-probabilities for all outcomes where the target qubit was |1>,
                # the total 'p_exc' is appended to the 'probabilities' list. np.clip is used to
                # force the value into the valid range [0, 1], as mitigation can sometimes produce values outside this range.

            except Exception as e:
            # Catches any exception during mitigation or probability calculation.
                 print(f"An error occurred during mitigation or probability calculation for circuit {i} at time {times[i]:.2f}: {e}")
                 import traceback
                 traceback.print_exc()
                 # Prints error message and traceback.
                 probabilities.append(np.nan) # Append NaN
                 # Appends NaN to the list.

        print("✓ Results processed and probabilities calculated.")
        # Prints success message after processing all circuit results.

except Exception as e:
# Catches any exception that occurred during the entire 'with Session(...)' block.
    print(f"An error occurred during the runtime session: {e}")
    import traceback
    traceback.print_exc()
    # Prints error message and traceback for session-level errors.
    # Ensure probabilities is empty if job fails completely
    probabilities = []
    # Clears the probabilities list if a session-level error occurred, as the data might be incomplete or corrupted.
    # Optionally re-raise the exception if you want it to stop the script execution
    # raise
   

# The session automatically closes here
# method is automatically called upon exiting the block, releasing resources.

# ------------------------------------------------------------------
#  Plot
# ------------------------------------------------------------------
# This section handles plotting the results.

# Filter out potential NaN values if any circuits failed processing
plot_times = [times[i] for i, p in enumerate(probabilities) if not np.isnan(p)]
# Creates a new list of time points, including only those for which a valid
# (non-NaN) probability was successfully calculated.
plot_probabilities = [p for p in probabilities if not np.isnan(p)]
# Creates a new list of probabilities, including only the valid (non-NaN) values.

if plot_probabilities: # Only plot if valid probabilities were successfully calculated
# Checks if the list of valid probabilities is not empty.
    plt.figure(figsize=(8,5))
    # Creates a new matplotlib figure with a specified size.
    plt.plot(plot_times, plot_probabilities, "o-",
             label=f"site {plot_site_qubit_index}, m={trotter_steps}")
    # Plots the valid probabilities against their corresponding time points.
    # "o-" creates points connected by lines. Sets a label for the legend.
    plt.xlabel("time (1/J)"); plt.ylabel("Probability") # 'spin-down' corresponds to |1> state
    # Sets the labels for the x and y axes. 
    plt.title("Hardware: spin down Probability of Site 0 vs. Time")
    # Sets the title of the plot.
    plt.ylim(-0.05, 1.05); plt.grid(); plt.legend(); plt.tight_layout()
    # Sets the y-axis limits, adds a grid, displays the legend, adjusts layout to
    # prevent overlaps, and prepares the plot for display.
    plt.show()
    # Displays the generated plot.
else:
# If no valid probabilities were calculated (list is empty after filtering NaNs).
    print("No valid probabilities were calculated. Plotting skipped.")
    # Prints a message indicating that plotting could not be done.
#--------end of script------------